# IndiaData One-Click Colab Runner
Edit only the config cell below, then run the run cell once.

In [ ]:
# CONFIG (edit these 4 values only)
REPO_URL = "https://github.com/abhinavjnu/IndiaData-colab.git"
REPO_BRANCH = "main"
API_KEY = "REPLACE_WITH_YOUR_API_KEY"
TARGET_YEARS = []  # Example: ['2023-24']; keep [] for auto-detect all

In [ ]:
import glob
import os
import re
import shutil
import subprocess
from datetime import datetime

def run(cmd, check=True):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")

def ensure_drive_mounted():
    try:
        from google.colab import drive
    except Exception as e:
        raise RuntimeError("This notebook must run in Google Colab.") from e
    if not os.path.exists('/content/drive'):
        os.makedirs('/content/drive', exist_ok=True)
    drive.mount('/content/drive', force_remount=False)

def detect_drive_root():
    candidates = ['/content/drive/MyDrive', '/content/drive/My Drive']
    for c in candidates:
        if os.path.exists(c):
            return c
    raise RuntimeError("Could not find Google Drive root (MyDrive/My Drive).")

def link_file(src, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.lexists(dest):
        os.remove(dest)
    os.symlink(src, dest)

def pick_first(paths):
    return paths[0] if paths else None

def map_year_from_drive(year_dir, repo_year_dir):
    txt_candidates = glob.glob(os.path.join(year_dir, '**', '*.TXT'), recursive=True)
    txt_candidates += glob.glob(os.path.join(year_dir, '**', '*.txt'), recursive=True)

    person = [p for p in txt_candidates if re.search(r'(cperv|person|individual)', os.path.basename(p), re.I)]
    hh = [p for p in txt_candidates if re.search(r'(chhv|household|hh)', os.path.basename(p), re.I)]

    xlsx_candidates = glob.glob(os.path.join(year_dir, '*.xlsx')) + glob.glob(os.path.join(year_dir, '*.XLSX'))
    layout = pick_first([p for p in xlsx_candidates if re.search(r'layout', os.path.basename(p), re.I)])
    if layout is None:
        layout = pick_first(xlsx_candidates)

    if not person or not layout:
        return False, 'Missing person TXT or layout XLSX'

    raw_dest = os.path.join(repo_year_dir, 'raw')
    os.makedirs(raw_dest, exist_ok=True)

    link_file(person[0], os.path.join(raw_dest, os.path.basename(person[0])))
    if hh:
        link_file(hh[0], os.path.join(raw_dest, os.path.basename(hh[0])))
    link_file(layout, os.path.join(repo_year_dir, os.path.basename(layout)))

    return True, f"Linked person={os.path.basename(person[0])}, hh={os.path.basename(hh[0]) if hh else 'none'}, layout={os.path.basename(layout)}"

def map_flat_raw(flat_raw_dir, repo_plfs_dir):
    person = pick_first(glob.glob(os.path.join(flat_raw_dir, 'CPERV*.TXT')) + glob.glob(os.path.join(flat_raw_dir, 'CPERV*.txt')) + glob.glob(os.path.join(flat_raw_dir, '*person*.TXT')) + glob.glob(os.path.join(flat_raw_dir, '*person*.txt')))
    hh = pick_first(glob.glob(os.path.join(flat_raw_dir, 'CHHV*.TXT')) + glob.glob(os.path.join(flat_raw_dir, 'CHHV*.txt')) + glob.glob(os.path.join(flat_raw_dir, '*household*.TXT')) + glob.glob(os.path.join(flat_raw_dir, '*household*.txt')))
    layout = pick_first(glob.glob(os.path.join(flat_raw_dir, '*Layout*.xlsx')) + glob.glob(os.path.join(flat_raw_dir, '*layout*.xlsx')) + glob.glob(os.path.join(flat_raw_dir, '*.xlsx')))

    if not person or not layout:
        return False, 'Flat raw folder missing person TXT or layout XLSX'

    year = '2024'
    m = re.search(r'(20[0-9]{2}-[0-9]{2,4})', os.path.basename(layout))
    if m:
        year = m.group(1)

    repo_year_dir = os.path.join(repo_plfs_dir, year)
    os.makedirs(os.path.join(repo_year_dir, 'raw'), exist_ok=True)

    link_file(person, os.path.join(repo_year_dir, 'raw', os.path.basename(person)))
    if hh:
        link_file(hh, os.path.join(repo_year_dir, 'raw', os.path.basename(hh)))
    link_file(layout, os.path.join(repo_year_dir, os.path.basename(layout)))

    return True, f"Mapped flat raw into year {year}"

# ---- One-click run starts here ----
ensure_drive_mounted()
drive_root = detect_drive_root()
base_dir = os.path.join(drive_root, 'IndiaDataData')

for d in ['raw', 'processed', 'codebooks', 'outputs/tables', 'outputs/figures', 'logs', 'PLFS']:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

print(f"Using Drive base: {base_dir}")

# Install system dependencies
run('apt-get -qq update')
run('apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev libfontconfig1-dev git')

# Clone fresh code each run
repo_dir = '/content/IndiaData'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
run(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {repo_dir}')

repo_plfs_dir = os.path.join(repo_dir, 'PLFS')
drive_plfs_root = os.path.join(base_dir, 'PLFS')

# Map Drive data into repo structure expected by automated_plfs_analysis.R
mapped = []
failed = []

year_dirs = [d for d in glob.glob(os.path.join(drive_plfs_root, '*')) if os.path.isdir(d)]
year_dirs = [d for d in year_dirs if re.search(r'20[0-9]{2}-[0-9]{2,4}$', os.path.basename(d))]

if TARGET_YEARS:
    target_set = set(TARGET_YEARS)
    year_dirs = [d for d in year_dirs if os.path.basename(d) in target_set]

for yd in sorted(year_dirs):
    year = os.path.basename(yd)
    ok, msg = map_year_from_drive(yd, os.path.join(repo_plfs_dir, year))
    (mapped if ok else failed).append((year, msg))

# Fallback: flat raw folder (single-year style)
if not mapped:
    ok, msg = map_flat_raw(os.path.join(base_dir, 'raw'), repo_plfs_dir)
    if ok:
        mapped.append(('flat-raw', msg))
    else:
        failed.append(('flat-raw', msg))

print('\nData mapping summary:')
for y, m in mapped:
    print(f'  [OK] {y}: {m}')
for y, m in failed:
    print(f'  [WARN] {y}: {m}')

if not mapped:
    raise RuntimeError("No usable PLFS data found in Drive. Place data in IndiaDataData/PLFS/<year>/... or IndiaDataData/raw/.")

# Write runtime config.yaml
config_yaml = f"""api:
  base_url: \"https://www.microdata.gov.in/NADA/index.php/api/\"
  api_key: \"{API_KEY}\"

paths:
  raw_data: \"{base_dir}/raw\"
  processed_data: \"{base_dir}/processed\"
  codebooks: \"{base_dir}/codebooks\"
  tables: \"{base_dir}/outputs/tables\"
  figures: \"{base_dir}/outputs/figures\"

settings:
  default_survey: \"plfs\"
  save_format: \"parquet\"

surveys:
  plfs:
    source: \"microdata.gov.in\"
"""

config_path = os.path.join(repo_dir, 'config.yaml')
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_yaml)

# Install R dependencies
run(f'Rscript {os.path.join(repo_dir, "R/00_setup.R")}', check=True)

# Execute analysis with timestamped log
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = os.path.join(base_dir, 'logs', f'plfs_run_{stamp}.log')
run(f'cd {repo_dir} && Rscript PLFS/automated_plfs_analysis.R 2>&1 | tee "{log_path}"', check=True)

print('\nRun complete.')
print('Log:', log_path)
print('Tables:', os.path.join(base_dir, 'outputs', 'tables'))
print('Figures:', os.path.join(base_dir, 'outputs', 'figures'))